# LFB Incident Records: Data Cleaning & Validation

**Business question:** Can the London Fire Brigade incident data be trusted for reporting response times by borough, and what had to be fixed first?

**Approach (QA-style):** profile the raw data, run validation checks grouped by data-quality dimension (completeness, uniqueness, validity, consistency, accuracy), clean with documented decisions, re-run the same checks, and report before vs after.

> TODO markers show where you add your own findings. Run top to bottom.

In [17]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path("../data/raw/lfb_incident_records.csv")   # TODO: change to your downloaded file name
CLEAN_PATH = Path("../data/clean/lfb_incidents_clean.csv")
LOG_PATH = Path("../reports/validation_summary.csv")

# Column names as I expect them in the LFB file. TODO: compare with raw.columns and edit if different.
COLS = {
    "id": "IncidentNumber",
    "date": "DateOfCall",
    "year": "CalYear",
    "hour": "HourOfCall",
    "group": "IncidentGroup",
    "borough": "IncGeo_BoroughName",
    "lat": "Latitude",
    "lon": "Longitude",
    "first_pump": "FirstPumpArriving_AttendanceTime",
    "second_pump": "SecondPumpArriving_AttendanceTime",
}

LAT_RANGE = (51.2, 51.8)      # generous bounding box around Greater London
LON_RANGE = (-0.6, 0.4)
EXPECTED_GROUPS = {"Fire", "Special Service", "False Alarm"}
MAX_ATTENDANCE_SECONDS = 3600  # over 60 minutes is flagged for review, not deleted
UK_LAT_RANGE = (49.0, 61.0)   # coordinates outside the UK are impossible for LFB incidents
UK_LON_RANGE = (-9.0, 2.0)


def parse_dates(series):
    # Parse dates explicitly (no day-first guessing). The LFB file uses YYYY-MM-DD.
    # Other formats are tried only for values the first format could not read.
    if pd.api.types.is_datetime64_any_dtype(series):
        return series
    s = series.astype("string").str.strip()
    out = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    for fmt in ["%Y-%m-%d %H:%M:%S", "%d/%m/%Y", "%d %b %Y", "%d-%b-%Y"]:
        todo = out.isna() & s.notna()
        if not todo.any():
            break
        out = out.fillna(pd.to_datetime(s.where(todo), format=fmt, errors="coerce"))
    return out

## 1. Load the raw data
If you downloaded several yearly files, read each one and combine them with `pd.concat` before profiling. Keep the original file untouched in `data/raw/`.

In [18]:
raw = pd.read_csv(RAW_PATH, low_memory=False)
print(f"Rows: {len(raw):,} | Columns: {raw.shape[1]}")
missing_cols = [v for v in COLS.values() if v not in raw.columns]
print("Expected columns not found (their checks will be skipped):", missing_cols)
raw.head()

Rows: 358,338 | Columns: 39
Expected columns not found (their checks will be skipped): []


,IncidentNumber,DateOfCall,CalYear,TimeOfCall,HourOfCall,IncidentGroup,StopCodeDescription,SpecialServiceType,PropertyCategory,PropertyType,...,FirstPumpArriving_AttendanceTime,FirstPumpArriving_DeployedFromStation,SecondPumpArriving_AttendanceTime,SecondPumpArriving_DeployedFromStation,NumStationsWithPumpsAttending,NumPumpsAttending,PumpCount,PumpMinutesRounded,Notional Cost (£),NumCalls
0,000002-01012024,2024-01-01,2024,00:00:43,0,Special Service,Special Service,Advice Only,Dwelling,Unlicensed House in Multiple Occupation - Up t...,...,172.0,Stratford,289.0,Stratford,2.0,2.0,2,60,388,1.0
1,000003-01012024,2024-01-01,2024,00:01:11,0,Special Service,Special Service,No action (not false alarm),Dwelling,Purpose Built Flats/Maisonettes - 10 or more s...,...,332.0,Shoreditch,NaN,NaN,1.0,1.0,1,60,388,1.0
2,000007-01012024,2024-01-01,2024,00:04:21,0,False Alarm,False alarm - Good intent,NaN,Outdoor,Loose refuse,...,521.0,Lambeth,NaN,NaN,1.0,1.0,1,60,388,1.0
3,000010-01012024,2024-01-01,2024,00:05:40,0,Special Service,Special Service,Lift Release,Dwelling,Purpose Built Flats/Maisonettes - 4 to 9 storeys,...,NaN,NaN,NaN,NaN,1.0,1.0,1,60,388,1.0
4,000011-01012024,2024-01-01,2024,00:06:32,0,Special Service,Special Service,Effecting entry/exit,Dwelling,Purpose Built Flats/Maisonettes - 10 or more s...,...,323.0,Lee Green,NaN,NaN,1.0,1.0,1,60,388,2.0


## 2. Profile the raw data

In [19]:
profile = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_pct": (raw.isna().mean() * 100).round(2),
    "unique_values": raw.nunique(),
})
profile.sort_values("missing_pct", ascending=False)

,dtype,missing,missing_pct,unique_values
SecondPumpArriving_AttendanceTime,float64,226712,63.27,1197
SecondPumpArriving_DeployedFromStation,str,226712,63.27,104
Northing_m,float64,221313,61.76,31517
Easting_m,float64,221313,61.76,40095
Postcode_full,str,221313,61.76,49763
Longitude,float64,221313,61.76,128217
Latitude,float64,221313,61.76,128186
SpecialServiceType,str,212744,59.37,20
FirstPumpArriving_DeployedFromStation,str,18655,5.21,104
FirstPumpArriving_AttendanceTime,float64,18654,5.21,1192


### 2b. Context for the biggest gaps
Before calling something an error, check whether it has an innocent explanation.

In [20]:
dates = parse_dates(raw[COLS["date"]])
print("Date range:", dates.min().date(), "to", dates.max().date())

if COLS["lat"] in raw.columns:
    no_loc = raw[COLS["lat"]].isna()
    print(f"Rows with no Latitude: {no_loc.mean() * 100:.1f}%")
    for col in ["Easting_rounded", "Northing_rounded", "Postcode_district", COLS["borough"]]:
        if col in raw.columns:
            print(f"  ...of those, rows that still have {col}: {raw.loc[no_loc, col].notna().mean() * 100:.1f}%")

if COLS["first_pump"] in raw.columns and COLS["group"] in raw.columns:
    by_group = raw.groupby(COLS["group"])[COLS["first_pump"]].apply(lambda x: x.isna().mean() * 100).round(1)
    print("\n% missing first pump arrival time, by incident group:")
    print(by_group.to_string())

Date range: 2024-01-01 to 2026-07-31
Rows with no Latitude: 61.8%
  ...of those, rows that still have Easting_rounded: 100.0%
  ...of those, rows that still have Northing_rounded: 100.0%
  ...of those, rows that still have Postcode_district: 100.0%
  ...of those, rows that still have IncGeo_BoroughName: 100.0%

% missing first pump arrival time, by incident group:
IncidentGroup
False Alarm         0.8
Fire                1.1
Special Service    11.4


**TODO: what does 2b tell you?** For example: is missing latitude explained by other location columns being filled? Is missing pump time concentrated in one incident group (suggesting a legitimate reason)? Check the Metadata file you downloaded.

## 3. Validation checks
One function runs every check and returns a table, so the same checks can be re-run on the cleaned data. Checks are skipped if a column is missing.

In [21]:
def run_checks(df, stage):
    rows = []
    n = len(df)

    def add(dimension, check, count):
        rows.append({
            "stage": stage,
            "dimension": dimension,
            "check": check,
            "rows_affected": int(count),
            "pct_of_rows": round(100 * count / n, 2) if n else 0,
        })

    def has(key):
        return COLS[key] in df.columns

    # Completeness
    for key in ["id", "date", "borough", "lat", "lon", "first_pump"]:
        if has(key):
            add("Completeness", f"Missing {COLS[key]}", df[COLS[key]].isna().sum())

    # Uniqueness
    add("Uniqueness", "Fully duplicated rows", df.duplicated().sum())
    if has("id"):
        add("Uniqueness", f"Repeated {COLS['id']}", df.duplicated(subset=COLS["id"]).sum())

    # Validity
    if has("date"):
        d = parse_dates(df[COLS["date"]])
        add("Validity", "Unparseable dates", (d.isna() & df[COLS["date"]].notna()).sum())
        add("Validity", "Dates in the future", (d > pd.Timestamp.today()).sum())
        if has("year"):
            yr = pd.to_numeric(df[COLS["year"]], errors="coerce")
            add("Consistency", "Year of DateOfCall differs from CalYear", (d.notna() & yr.notna() & (d.dt.year != yr)).sum())
    if has("hour"):
        h = pd.to_numeric(df[COLS["hour"]], errors="coerce")
        add("Validity", "Hour outside 0-23", ((h < 0) | (h > 23)).sum())
    if has("lat") and has("lon"):
        la = pd.to_numeric(df[COLS["lat"]], errors="coerce")
        lo = pd.to_numeric(df[COLS["lon"]], errors="coerce")
        inside = la.between(*LAT_RANGE) & lo.between(*LON_RANGE)
        add("Validity", "Coordinates outside Greater London box", (la.notna() & lo.notna() & ~inside).sum())
    for key in ["first_pump", "second_pump"]:
        if has(key):
            t = pd.to_numeric(df[COLS[key]], errors="coerce")
            add("Validity", f"Negative {COLS[key]}", (t < 0).sum())
            add("Validity", f"{COLS[key]} over {MAX_ATTENDANCE_SECONDS // 60} min (review)", (t > MAX_ATTENDANCE_SECONDS).sum())

    # Consistency
    if has("borough"):
        s = df[COLS["borough"]].dropna().astype(str)
        spellings = s.groupby(s.str.strip().str.upper()).transform("nunique")
        add("Consistency", "Rows with borough spelling/case variants", (spellings > 1).sum())
    if has("group"):
        g = df[COLS["group"]]
        add("Consistency", "Unexpected IncidentGroup values", (g.notna() & ~g.isin(EXPECTED_GROUPS)).sum())

    # Accuracy / plausibility
    if has("first_pump") and has("second_pump"):
        a = pd.to_numeric(df[COLS["first_pump"]], errors="coerce")
        b = pd.to_numeric(df[COLS["second_pump"]], errors="coerce")
        add("Accuracy", "Second pump arrived before first pump", (a.notna() & b.notna() & (b < a)).sum())

    return pd.DataFrame(rows)


before = run_checks(raw, "Raw")
before

,stage,dimension,check,rows_affected,pct_of_rows
0,Raw,Completeness,Missing IncidentNumber,0,0.00
1,Raw,Completeness,Missing DateOfCall,0,0.00
2,Raw,Completeness,Missing IncGeo_BoroughName,0,0.00
3,Raw,Completeness,Missing Latitude,221313,61.76
4,Raw,Completeness,Missing Longitude,221313,61.76
5,Raw,Completeness,Missing FirstPumpArriving_AttendanceTime,18654,5.21
6,Raw,Uniqueness,Fully duplicated rows,0,0.00
7,Raw,Uniqueness,Repeated IncidentNumber,0,0.00
8,Raw,Validity,Unparseable dates,0,0.00
9,Raw,Validity,Dates in the future,0,0.00


**TODO: findings from the raw checks.** Which problems are biggest, and which would distort a response-time report? (2 to 3 sentences.)

## 4. Cleaning (with a decision log)
Rule of thumb used here: **fix** what can be fixed safely, **flag** what is suspicious but possibly real, and only **drop** exact duplicates. Every step is logged with a reason.

In [22]:
clean = raw.copy()
cleaning_log = []

def log_step(step, rows_changed, why):
    cleaning_log.append({"step": step, "rows_changed": int(rows_changed), "why": why})

# 4.1 Trim stray whitespace in text columns
trimmed_cells = 0
trim_report = {}
for col in clean.columns:
    if pd.api.types.is_string_dtype(clean[col]) or pd.api.types.is_object_dtype(clean[col]):
        try:
            stripped = clean[col].str.strip()
        except AttributeError:      # column holds no text values, nothing to trim
            continue
        n_trim = int((clean[col].notna() & (clean[col] != stripped)).sum())
        if n_trim:
            trim_report[col] = n_trim
        trimmed_cells += n_trim
        clean[col] = stripped
log_step("Trim whitespace in text columns", trimmed_cells, "Prevents the same value being counted as two categories (count = cells changed)")

# 4.2 Standardise borough names to the most common spelling per name
b = COLS["borough"]
if b in clean.columns:
    key = clean[b].str.upper()
    canonical = clean.groupby(key)[b].agg(lambda x: x.value_counts().idxmax())
    new_b = key.map(canonical)
    changed = (clean[b].notna() & (clean[b] != new_b)).sum()
    clean[b] = new_b
    log_step("Standardise borough names", changed, "One spelling per borough so grouping and joins work")

# 4.3 Parse dates
d = COLS["date"]
if d in clean.columns:
    parsed = parse_dates(clean[d])
    unparsed = int((parsed.isna() & clean[d].notna()).sum())
    log_step("Parse DateOfCall to datetime", parsed.notna().sum(), "Explicit YYYY-MM-DD parsing; real dates allow time-based analysis")
    if unparsed:
        log_step("DateOfCall values that could not be parsed (now missing)", unparsed, "Left as missing rather than guessed; TODO: inspect these")
    clean[d] = parsed

In [23]:
# 4.4 Remove duplicates
n0 = len(clean)
clean = clean.drop_duplicates()
log_step("Drop fully duplicated rows", n0 - len(clean), "Exact copies add no information and inflate counts")

if COLS["id"] in clean.columns:
    n0 = len(clean)
    clean = clean.drop_duplicates(subset=COLS["id"], keep="first")
    log_step("Drop repeated IncidentNumber (keep first)", n0 - len(clean), "One row per incident; TODO: confirm repeats were true duplicates, not different records")

# 4.5 Coordinates: fix the impossible, flag the unusual
la, lo = COLS["lat"], COLS["lon"]
if la in clean.columns and lo in clean.columns:
    clean[la] = pd.to_numeric(clean[la], errors="coerce")
    clean[lo] = pd.to_numeric(clean[lo], errors="coerce")
    has_xy = clean[la].notna() & clean[lo].notna()
    in_uk = clean[la].between(*UK_LAT_RANGE) & clean[lo].between(*UK_LON_RANGE)
    impossible = has_xy & ~in_uk
    clean.loc[impossible, [la, lo]] = np.nan
    log_step("Set coordinates outside the UK to missing", impossible.sum(), "Cannot be a real LFB location (e.g. 0,0 or swapped values); the incident is kept")

    in_london = clean[la].between(*LAT_RANGE) & clean[lo].between(*LON_RANGE)
    outside = clean[la].notna() & clean[lo].notna() & ~in_london
    clean["coords_outside_london_flag"] = outside
    log_step("Flag coordinates outside Greater London box (kept)", outside.sum(), "LFB can attend incidents just outside London, so these may be genuine; TODO: inspect before deciding")

# 4.6 Attendance times: fix impossible values, flag unusual ones
for key in ["first_pump", "second_pump"]:
    col = COLS[key]
    if col in clean.columns:
        clean[col] = pd.to_numeric(clean[col], errors="coerce")
        neg = clean[col] < 0
        clean.loc[neg, col] = np.nan
        log_step(f"Set negative {col} to missing", neg.sum(), "A negative travel time is impossible")
        flag = clean[col] > MAX_ATTENDANCE_SECONDS
        clean[col + "_over_60min_flag"] = flag
        log_step(f"Flag {col} over 60 min", flag.sum(), "Could be a real long response, so kept and flagged for review")

fp, sp = COLS["first_pump"], COLS["second_pump"]
if fp in clean.columns and sp in clean.columns:
    bad = clean[fp].notna() & clean[sp].notna() & (clean[sp] < clean[fp])
    clean["second_before_first_flag"] = bad
    log_step("Flag second pump arriving before first", bad.sum(), "Logically impossible but cause unknown, so flagged for review instead of changed")

print("Cells trimmed per column:")
print(pd.Series(trim_report, name="cells_trimmed").sort_values(ascending=False).to_string())
pd.DataFrame(cleaning_log)

Cells trimmed per column:
PropertyType    300525


,step,rows_changed,why
0,Trim whitespace in text columns,300525,Prevents the same value being counted as two c...
1,Standardise borough names,0,One spelling per borough so grouping and joins...
2,Parse DateOfCall to datetime,358338,Explicit YYYY-MM-DD parsing; real dates allow ...
3,Drop fully duplicated rows,0,Exact copies add no information and inflate co...
4,Drop repeated IncidentNumber (keep first),0,One row per incident; TODO: confirm repeats we...
5,Set coordinates outside the UK to missing,542,"Cannot be a real LFB location (e.g. 0,0 or swa..."
6,Flag coordinates outside Greater London box (k...,0,"LFB can attend incidents just outside London, ..."
7,Set negative FirstPumpArriving_AttendanceTime ...,0,A negative travel time is impossible
8,Flag FirstPumpArriving_AttendanceTime over 60 min,0,"Could be a real long response, so kept and fla..."
9,Set negative SecondPumpArriving_AttendanceTime...,0,A negative travel time is impossible


**TODO: inspect the flagged coordinates.** Do the rows outside the London box have London borough names (suggesting bad coordinates) or not (suggesting genuine incidents just outside London)? Decide whether to keep, fix or drop, and say why.

In [24]:
if "coords_outside_london_flag" in clean.columns:
    flagged = clean[clean["coords_outside_london_flag"]]
    print(f"{len(flagged):,} flagged rows")
    print(flagged[[COLS["borough"], COLS["lat"], COLS["lon"]]].head(15).to_string())
    print()
    print(flagged[COLS["borough"]].value_counts().head(10).to_string())

0 flagged rows
Empty DataFrame
Columns: [IncGeo_BoroughName, Latitude, Longitude]
Index: []

Series([], )


## 5. Re-run the same checks on the clean data

In [25]:
after = run_checks(clean, "Clean")

summary = (
    pd.concat([before, after])
    .pivot_table(index=["dimension", "check"], columns="stage", values="rows_affected", aggfunc="sum")
    .reset_index()
)
summary["raw_minus_clean"] = summary["Raw"] - summary["Clean"]   # negative for Missing = bad values deliberately set to missing
print(f"Rows: raw {len(raw):,} -> clean {len(clean):,}")
summary

Rows: raw 358,338 -> clean 358,338


stage,dimension,check,Clean,Raw,raw_minus_clean
0,Accuracy,Second pump arrived before first pump,2296,2296,0
1,Completeness,Missing DateOfCall,0,0,0
2,Completeness,Missing FirstPumpArriving_AttendanceTime,18654,18654,0
3,Completeness,Missing IncGeo_BoroughName,0,0,0
4,Completeness,Missing IncidentNumber,0,0,0
5,Completeness,Missing Latitude,221855,221313,-542
6,Completeness,Missing Longitude,221855,221313,-542
7,Consistency,Rows with borough spelling/case variants,0,0,0
8,Consistency,Unexpected IncidentGroup values,0,0,0
9,Consistency,Year of DateOfCall differs from CalYear,0,0,0


**TODO: before/after story.** Copy the key numbers into the README, e.g. "X duplicate rows removed, Y invalid coordinates corrected, Z borough spellings merged into N boroughs".

## 6. Save outputs

In [26]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

clean.to_csv(CLEAN_PATH, index=False)
summary.to_csv(LOG_PATH, index=False)
pd.DataFrame(cleaning_log).to_csv(LOG_PATH.with_name("cleaning_log.csv"), index=False)
print("Saved:", CLEAN_PATH, LOG_PATH)

Saved: ..\data\clean\lfb_incidents_clean.csv ..\reports\validation_summary.csv


## 7. Limitations and next steps (TODO)
- What could not be fixed, and why?
- Which flagged records need a domain expert?
- A natural follow-up: use the clean data for a response-time dashboard by borough.